<a href="https://colab.research.google.com/github/Shemuel-bot/AI-ball-notebooks/blob/main/ai_ball_original.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import math
import copy

def TokenizeInput(input, alphabet):
    input_str = str(input)
    matches = []

    for order, item in enumerate(alphabet):
        item_str = str(item)
        if not item_str:
            continue

        # Loop to find ALL occurrences of the item instead of just the first
        start_pos = 0
        while True:
            index = input_str.find(item_str, start_pos)
            if index == -1:
                break

            matches.append((index, -len(item_str), order, item))

            # Move start position forward by 1 to continue searching the rest of the string
            # (Adding 1 instead of len(item_str) allows it to safely catch overlapping patterns)
            start_pos = index + 1

    # Sorts by starting index, then by longest string, then by original order
    matches.sort(key=lambda match: (match[0], match[1], match[2]))

    non_overlapping_items = []
    last_end_index = 0

    for match in matches:
        start_index = match[0]
        length = -match[1]  # The length was stored as a negative number
        item = match[3]

        # Only add the token if it doesn't overlap with the previously added token
        if start_index >= last_end_index:
            non_overlapping_items.append(item)
            last_end_index = start_index + length
    #print(non_overlapping_items)
    return non_overlapping_items

class Ball_Trap_Brain:
    def __init__(self, deflectors, bounce, alphabet, output, holes_multiplier=1):
        self.output = output
        self.holes_multiplier = holes_multiplier
        self.ball_energy = bounce*2
        self.alphabet = alphabet
        self.alphabet_directions = self.choose_alphabet_direction(len(alphabet))
        self.alphabet_start_positions = self.start_positions(len(alphabet), deflectors)
        self.deflectors_list = [[random.randint(0, deflectors-1), bounce] for i in range(deflectors)]
        self.set_holes(output, deflectors, self.holes_multiplier)

    def set_holes (self, output, deflectors, holes_multiplier):
        for i in range(len(output)):
          for j in range(holes_multiplier):
            self.deflectors_list[random.randint(0, deflectors-1)][1] = output[i]

    def choose_alphabet_direction(self, alphabet):
        directions = [random.choice([-1,1]) for _ in range(alphabet)]
        return directions

    def start_positions(self, alphabet, deflectors):
        return [random.randint(0, deflectors-1) for i in range(alphabet)]


    def compute_input(self, input):
        directions = []
        positions = []
        travel = []
        input_arr = TokenizeInput(input, self.alphabet)

        for i in input_arr:
            if i in self.alphabet:
                index = self.alphabet.index(i)
                directions.append(self.alphabet_directions[index])
                positions.append(self.alphabet_start_positions[index])

        for i in range(len(directions)):
            travel.append(self.travel(copy.copy(directions), positions[i], i))


        return self.ricochet(travel, 10, self.ball_energy)

    def ricochet(self, travel, cap, ball_energy):
        output = []
        dl = self.deflectors_list
        n = len(dl)

        for t in travel:
            # normalize index using modulo (handles large or negative values)
            new_idx = t % n
            cap_idx = 0
            be = ball_energy

            while be > 0 and cap_idx < cap:
                dest, val = dl[new_idx]
                distance = abs(dest - new_idx)

                if not isinstance(val, int):
                    output.append(val)
                    be += 1
                    be -= distance
                    new_idx = dest
                    cap_idx += 1
                    continue

                be += val
                be -= distance
                new_idx = dest
                cap_idx += 1

        return output

    def compute_deflectors(self, input):
        """
        Processes input exactly like compute_input but returns an array
        of all deflectors hit during the ball's travel.
        """
        directions = []
        positions = []
        travel = []
        input_arr = TokenizeInput(input, self.alphabet)

        for i in input_arr:
            if i in self.alphabet:
                index = self.alphabet.index(i)
                directions.append(self.alphabet_directions[index])
                positions.append(self.alphabet_start_positions[index])

        for i in range(len(directions)):
            travel.append(self.travel(copy.copy(directions), positions[i], i))

        # Passes the standard cap of 10 and the current ball_energy
        return self.ricochet_deflectors(travel, 10, self.ball_energy)

    def ricochet_deflectors(self, travel, cap, ball_energy):
        """
        Modified ricochet logic that records and returns the deflectors
        hit instead of the output values.
        """
        deflectors_hit = []
        dl = self.deflectors_list
        n = len(dl)

        for t in travel:
            # normalize index using modulo (handles large or negative values)
            new_idx = t % n
            cap_idx = 0
            be = ball_energy

            while be > 0 and cap_idx < cap:
                dest, val = dl[new_idx]

                # Append the deflector's details to our tracking array
                deflectors_hit.append(new_idx)

                distance = abs(dest - new_idx)

                if not isinstance(val, int):
                    be += 1
                    be -= distance
                    new_idx = dest
                    cap_idx += 1
                    continue

                be += val
                be -= distance
                new_idx = dest
                cap_idx += 1

        return deflectors_hit

    def travel(self, directions, position, index):
        directions.pop(index)
        position += sum(directions)
        return position

    def display_deflectors(self):
        for i, deflect in enumerate(self.deflectors_list):
            print(f"Deflector {i}: {deflect}")

    def clone(self):
      # Create a lightweight dummy instance
      new_brain = Ball_Trap_Brain(1, self.ball_energy // 2, self.alphabet, self.output, self.holes_multiplier)

      # Fast shallow copy of the 10,000 elements
      new_brain.deflectors_list = [[d, v] for d, v in self.deflectors_list]
      new_brain.alphabet_directions = self.alphabet_directions[:]
      new_brain.alphabet_start_positions = self.alphabet_start_positions[:]

      return new_brain

    def shift_alphabet(self):
        random.shuffle(self.alphabet)
        for i in range(len(self.deflectors_list)):
          if self.deflectors_list[i][1] in self.output:
            idx = self.output.index(self.deflectors_list[i][1])
            self.deflectors_list[i][1] = self.alphabet[idx]

    def Mutate(self, individual_probability=0.1, probability=0.05):
        if random.random() < individual_probability:
            for i in range(len(self.deflectors_list)):
                  if random.random() < probability:
                      self.deflectors_list[i][0] = random.randint(0, len(self.deflectors_list)-1)

            for i in range(len(self.deflectors_list)):
                 if random.random() < probability:
                     self.deflectors_list[random.randint(0, len(self.deflectors_list)-1)][1] = random.choice(self.output)

            for i in range(len(self.alphabet_directions)):
                 if random.random() < probability:
                     self.alphabet_directions[i] = 1 if self.alphabet_directions[i] == -1 else -1

            for i in range(len(self.alphabet_start_positions)):
                  if random.random() < probability:
                      self.alphabet_start_positions[i] = random.randint(0, len(self.deflectors_list)-1)

            if random.random() < probability:
                  self.set_holes(self.output, len(self.deflectors_list), self.holes_multiplier)

        return self

In [ ]:
import random
import string
def generate_test_cases(amount, number, indiv=True):

  test_cases = []
  if indiv == True:
    test_cases = [(f'1+{i}', f"{random.randint(0, 1000)}") for i in range(amount)]
  else:
    test_cases = [(f'1+{i}', f'{1+i}') for i in range(amount)]

  return test_cases

In [ ]:

"""
Fitness evaluation module for Ball_Trap_Brain genetic algorithm.

This module contains functions to evaluate the performance of individual brains
and entire populations against test cases using string similarity metrics.
"""

import difflib
import math


def evaluate_performance(input_val, answer, brain, ratio=False):

    output = brain.compute_input(input_val)
    if ratio:
        return difflib.SequenceMatcher(None, ''.join(output), answer).ratio()
    else:
      return 1 if ''.join(output) == answer else 0


import concurrent.futures

def eval_single_brain(args):
    individual, test_cases, ratio = args
    avg_fitness = sum(
        evaluate_performance(inp, ans, individual, ratio) for inp, ans in test_cases
    ) / len(test_cases)
    return avg_fitness

def evaluate_population_parallel(population, test_cases, ratio=False):
    # Package arguments for the worker pool
    args = [(ind, test_cases, ratio) for ind in population]

    # Run across available CPU cores
    with concurrent.futures.ProcessPoolExecutor() as executor:
        results = list(executor.map(eval_single_brain, args))

    best_fitness = max(results)
    best_brain = population[results.index(best_fitness)]
    return best_fitness, best_brain

In [ ]:
"""
Genetic algorithm to evolve Ball_Trap_Brain instances to produce desired outputs.

This module runs a genetic algorithm that evolves a population of brains over
multiple generations. Each brain is evaluated against test cases, and the best
performer is mutated to create the next generation.
"""

string = "This module runs a genetic algorithm that evolves a population of brains over multiple generations. Each brain is evaluated against test cases, and the best performer is mutated to create the next generation."

import copy
import difflib
import random
import shelve

# Test cases: pairs of (input_string, expected_output_string) used for fitness evaluation


alphabet = [f'{i}' for i in range(10)] + ['+'] #+ list("abcdefghijklmnopqrstuvwxyz")  # Define the alphabet used in the brains
output = [f'{i}' for i in range(10)]

def main(mini_batch, alphabet, generations=500, pop_size=100, brain=Ball_Trap_Brain(1000, 4, alphabet, output, 2), ratio=False):
    """
    Run the genetic algorithm for evolving Ball_Trap_Brain instances.

    Returns:
        Ball_Trap_Brain: The best brain found after evolution.
    """
    # Evolution hyperparameters
    population_size = pop_size

    # Initialize population with random brains
    # Parameters: 10 deflectors, 2 bounce energy, alphabet of 6 characters

    population = [brain.clone().Mutate(1.0, 0.025) for _ in range(population_size)]
    population.append(brain)
    best_individual = None
    mini_batch_0 = copy.copy(mini_batch)
    #a = [mini_batch_0.pop(0) for i in range(10)]
    highest_score = -1
    # Run evolution loop
    for generation in range(generations):

        # Evaluate all brains and find the best performer
        best_fitness, best_individual = evaluate_population_parallel(population,mini_batch, ratio)



        # Check for perfect solution
        if best_fitness >= 1:
          print(f"Generation {generation}: Solution found!")
          return best_individual
          #for i in range(10):
            #a.append(mini_batch_0.pop(0))


        # Create next generation by mutating the best brain multiple times
        # New code: passes best_fitness, individual_probability, base_probability
        new_population = [best_individual.clone().Mutate(1.0, 0.025) for _ in range(population_size)]
        population = new_population
        population.append(best_individual)
        print(f"Generation {generation}: Best fitness = {best_fitness}")
    return best_individual

In [ ]:
from collections import Counter

class Council_Brain:
    def __init__(self, brains):
        """
        Initializes the council with a list of separately trained brains.
        """
        self.brains = brains

    def compute_input(self, input_val):
        """
        Each brain processes the input separately.
        The council votes on the final answer by finding the most common output.
        """
        votes = []
        for brain in self.brains:
            # Each brain outputs a list of characters; join them to form the string
            output_list = brain.compute_input(input_val)
            output_str = ''.join(output_list)
            votes.append(output_str)

        # Handle empty cases securely
        if not votes:
            return []

        # Tally the votes and find the majority
        vote_counts = Counter(votes)
        majority_vote = vote_counts.most_common(1)[0][0]

        # Return as a list of characters to perfectly match the original Ball_Trap_Brain format
        return list(majority_vote)

def train_council(num_brains, training_data, alphabet, output_format, generations=50, pop_size=100):
    """
    Trains a council where each brain is trained separately using the genetic algorithm.
    """
    trained_brains = []

    for i in range(num_brains):
        print(f"\n=====================================")
        print(f" Training Council Brain {i+1} / {num_brains}")
        print(f"=====================================")
        base_brain = Ball_Trap_Brain(100, 10, alphabet, output, 3)
        # Initialize a fresh brain for this specific council member
        #base_brain = Ball_Trap_Brain(100, 10, alphabet, output_format, 2)

        # Train separately using the existing main GA loop
        for j in range(1):
          print(f'---{j}---')
          base_brain = main(
              random.sample(training_data,5),
              alphabet=alphabet,
              generations=generations,
              pop_size=pop_size,
              brain=base_brain,
              ratio=True
          )
        trained_brains.append(base_brain)

    return Council_Brain(trained_brains)

In [ ]:
if __name__ == '__main__':
    # Generate the test cases using the existing generation function
    test_cases = generate_test_cases(amount=200, number='1', indiv=False)
    random.shuffle(test_cases)

    # Split the dataset
    training_0 = test_cases[:int(0.8 * len(test_cases))]
    training_test = test_cases[int(0.8 * len(test_cases)):]


    # Define council parameters
    COUNCIL_SIZE = 200     # Number of voting brains in the council
    GENERATIONS = 20     # Generations per brain
    POP_SIZE = 100        # Population size per generation

    # Train the council (each brain is trained entirely separately)
    print(f"Initializing Council Training with {COUNCIL_SIZE} brains...")
    council = train_council(
        num_brains=COUNCIL_SIZE,
        training_data=training_0,
        alphabet=alphabet,
        output_format=output,
        generations=GENERATIONS,
        pop_size=POP_SIZE
    )



    # Evaluate the entire council's performance against the testing set
    # Note: evaluate_population_parallel relies on a list of brains, so we wrap the council in a list
    print("\nEvaluating Council performance on test batch...")
    council_fitness, _ = evaluate_population_parallel([council], training_test, False)
    council_fitness_training, _ = evaluate_population_parallel([council], training_0, False)
    print(f"Overall Council Fitness: {council_fitness}, Council training fitness: {council_fitness_training}")
    # while True:
    #   print(council.compute_input(input("Q: ")))

Streaming output truncated to the last 5000 lines.
---0---
Generation 0: Best fitness = 0.4314285714285714
Generation 1: Best fitness = 0.48
Generation 2: Best fitness = 0.5142857142857142
Generation 3: Best fitness = 0.5642857142857143
Generation 4: Best fitness = 0.6333333333333333
Generation 5: Best fitness = 0.6333333333333333
Generation 6: Best fitness = 0.6714285714285714
Generation 7: Best fitness = 0.6714285714285714
Generation 8: Best fitness = 0.6714285714285714
Generation 9: Best fitness = 0.6714285714285714
Generation 10: Best fitness = 0.6714285714285714
Generation 11: Best fitness = 0.7285714285714285
Generation 12: Best fitness = 0.7285714285714285
Generation 13: Best fitness = 0.7285714285714285
Generation 14: Best fitness = 0.7285714285714285
Generation 15: Best fitness = 0.7285714285714285
Generation 16: Best fitness = 0.7285714285714285
Generation 17: Best fitness = 0.7285714285714285
Generation 18: Best fitness = 0.7285714285714285
Generation 19: Best fitness = 0.72